# Deploy EXACT 2026 Physics API With Qwen2.5 3B Adapters

This notebook starts the EXACT 2026 physics API on Kaggle. It serves Qwen2.5-3B-Instruct with three LoRA adapters: `numeric_parser_final`, `trace_explainer_final`, and `chlt_reasoner_final`, then exposes `/predict` through ngrok.

Expected Kaggle inputs:
- The API package folder containing `physics_api_server.py`, `physics_pipeline.py`, `physics_engine_core.py`, and `physics_calculator_v2`.
- Adapter folders for numeric parser, trace explainer, and CHLT reasoner.
- The verified runtime dataset `verified_golden_expanded.csv`.


## 1. Install Runtime Packages

Install the serving and API dependencies. Kaggle is Linux-based, so vLLM can run here while it may fail on local Windows. Run this before the grading slot if package installation is slow.


In [1]:
!pip -q install -U vllm flask flask-cors pyngrok pandas scikit-learn numpy z3-solver
# Kaggle T4 + recent vLLM may auto-select FlashInfer attention, which has been unstable here.
# Remove optional FlashInfer packages so vLLM falls back to a safer attention backend.
!pip -q uninstall -y flashinfer flashinfer-python flashinfer-cubin


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 MB 6.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 96.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

## 2. GPU Preflight Check

This API needs a CUDA GPU because vLLM serves the Qwen base model and LoRA adapters. If this cell fails, enable a Kaggle GPU accelerator before starting vLLM.


In [2]:
import subprocess, sys, os, glob
from pathlib import Path

print('Checking CUDA/GPU visibility...')
try:
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=20)
    print(smi.stdout[-4000:] if smi.stdout else smi.stderr[-4000:])
    if smi.returncode != 0:
        raise RuntimeError('nvidia-smi failed. Kaggle GPU accelerator is probably not enabled.')
except FileNotFoundError:
    raise RuntimeError('nvidia-smi was not found. Kaggle GPU accelerator is not enabled.')

import torch
print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError('CUDA is not available. In Kaggle, open Settings -> Accelerator -> GPU, then restart the session and run again.')

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f'GPU {idx}: {props.name}, VRAM={props.total_memory/1024**3:.1f} GB, compute={props.major}.{props.minor}')

# Kaggle sometimes exposes libcuda.so.1 but not the unversioned libcuda.so needed by FlashInfer JIT linking.
# Create a writable symlink and prepend it to linker/runtime search paths.
candidates = []
for pattern in [
    '/usr/lib/x86_64-linux-gnu/libcuda.so*',
    '/usr/local/cuda/compat/libcuda.so*',
    '/usr/local/nvidia/lib64/libcuda.so*',
]:
    candidates.extend(glob.glob(pattern))

libcuda = None
for c in sorted(set(candidates)):
    if c.endswith('.so'):
        libcuda = c
        break
if libcuda is None:
    for c in sorted(set(candidates)):
        if c.endswith('.so.1') or '.so.' in c:
            libcuda = c
            break

if libcuda:
    cuda_link_dir = Path('/kaggle/working/cuda_link_lib')
    cuda_link_dir.mkdir(parents=True, exist_ok=True)
    target = cuda_link_dir / 'libcuda.so'
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(Path(libcuda))
    os.environ['LIBRARY_PATH'] = f'{cuda_link_dir}:{os.environ.get("LIBRARY_PATH", "")}'
    os.environ['LD_LIBRARY_PATH'] = f'{cuda_link_dir}:{os.environ.get("LD_LIBRARY_PATH", "")}'
    print('Created libcuda.so symlink:', target, '->', libcuda)
else:
    print('Warning: could not find libcuda.so/libcuda.so.1. FlashInfer JIT may fail.')

# T4 has compute capability 7.5, so avoid FlashAttention-2 and FlashInfer sampler paths when possible.
os.environ['VLLM_LOGGING_LEVEL'] = 'INFO'
# Kaggle T4 + vLLM V1 may pick FlashInfer attention, which is unstable here.
# Force the older engine and xFormers attention to avoid FlashInfer attention crashes.
os.environ['VLLM_USE_V1'] = '0'
os.environ['VLLM_ATTENTION_BACKEND'] = 'XFORMERS'
os.environ['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
os.environ.pop('FLASHINFER_DISABLE_JIT', None)
print('VLLM_USE_V1:', os.environ['VLLM_USE_V1'])
print('VLLM_ATTENTION_BACKEND:', os.environ['VLLM_ATTENTION_BACKEND'])
print('VLLM_USE_FLASHINFER_SAMPLER:', os.environ['VLLM_USE_FLASHINFER_SAMPLER'])
print('FLASHINFER_DISABLE_JIT:', os.environ.get('FLASHINFER_DISABLE_JIT'))
print('GPU preflight passed.')


Checking CUDA/GPU visibility...
Sun Jun 14 09:09:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------

## 3. Locate Runtime Files Automatically

This cell searches `/kaggle/input` for the API source files, LoRA adapters, and verified dataset. It does not depend on the Kaggle dataset name. If multiple copies exist, it prefers the clean package layout under `kaggle_api_package_v1/result`.


In [3]:
import os, sys, time, json, shutil, subprocess, textwrap
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/exact_physics_server')
WORK.mkdir(parents=True, exist_ok=True)

if not INPUT.exists():
    raise RuntimeError('/kaggle/input does not exist. Add the API package as Kaggle input data first.')

def _path_score(path, prefer_tokens=()):
    text = str(path).replace('\\', '/').lower()
    score = 0
    preferred = [
        'physics_api_package',
        'physics_api_package',
        '/result/',
        '/data/',
        'numeric_parser_final',
        'trace_explainer_final',
        'chlt_reasoner_final',
        '/adapter/',
    ]
    penalties = ['__pycache__', '/checkpoint-', '/checkpoints/', 'physics_planner', 'planner_adapter', 'old']
    for token in preferred:
        if token in text:
            score += 10
    for token in penalties:
        if token in text:
            score -= 50
    for token in prefer_tokens:
        if str(token).lower() in text:
            score += 25
    score -= len(path.parts) * 0.05
    return score

def find_file(filename, prefer_tokens=()):
    hits = [p for p in INPUT.rglob(filename) if p.is_file()]
    hits = sorted(hits, key=lambda p: (-_path_score(p, prefer_tokens), str(p)))
    if not hits:
        examples = '\n'.join(str(p) for p in list(INPUT.rglob('*'))[:30])
        raise FileNotFoundError(f'Could not find {filename} under /kaggle/input. First input entries:\n{examples}')
    return hits[0], hits

def _read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return {}

def find_adapter(kind):
    purpose_tokens = {
        'trace_explainer': ['trace_explainer_final', 'trace_explainer', 'explanation'],
        'numeric_parser': ['numeric_parser_final', 'numeric_parser'],
        'chlt_reasoner': ['chlt_reasoner_final', 'chlt_reasoner', 'chlt'],
    }[kind]
    candidates = []
    for cfg in INPUT.rglob('adapter_config.json'):
        adapter_dir = cfg.parent
        if not (adapter_dir / 'adapter_model.safetensors').exists():
            continue
        meta = _read_json(adapter_dir / 'training_metadata.json')
        parent_meta = _read_json(adapter_dir.parent / 'metadata.json')
        blob = ' '.join([
            str(adapter_dir).lower(),
            str(adapter_dir.parent).lower(),
            str(meta.get('adapter_purpose', '')).lower(),
            str(meta.get('run_name', '')).lower(),
            str(parent_meta.get('adapter_purpose', '')).lower(),
            str(parent_meta.get('adapter_name', '')).lower(),
        ])
        token_hits = sum(1 for t in purpose_tokens if t in blob)
        if token_hits == 0:
            continue
        score = _path_score(adapter_dir, purpose_tokens) + token_hits * 100
        if adapter_dir.name == 'adapter':
            score += 30
        candidates.append((score, adapter_dir, meta, parent_meta))
    candidates.sort(key=lambda x: (-x[0], str(x[1])))
    if not candidates:
        seen = '\n'.join(str(p.parent) for p in INPUT.rglob('adapter_config.json'))
        raise FileNotFoundError(f'Could not find {kind} LoRA adapter. Found adapter_config directories:\n{seen}')
    return candidates[0][1], candidates

server_file, server_hits = find_file('physics_api_server.py', ['physics_api_package', 'result'])
pipeline_file, pipeline_hits = find_file('physics_pipeline.py', ['physics_api_package', 'result'])
core_file, core_hits = find_file('physics_engine_core.py', ['physics_api_package', 'result'])
verified_file, verified_hits = find_file('verified_golden_expanded.csv', ['physics_api_package', 'data'])
trace_adapter_dir, trace_candidates = find_adapter('trace_explainer')
numeric_adapter_dir, numeric_candidates = find_adapter('numeric_parser')
chlt_adapter_dir, chlt_candidates = find_adapter('chlt_reasoner')

for src in [server_file, pipeline_file, core_file]:
    shutil.copy2(src, WORK / src.name)

calc_src = server_file.parent / 'physics_calculator_v2'
if not calc_src.exists():
    calc_src = next((p for p in INPUT.rglob('physics_calculator_v2') if p.is_dir() and (p / 'calculator.py').exists()), None)
if calc_src is None or not calc_src.exists():
    raise FileNotFoundError('Could not find physics_calculator_v2 package in the API package.')
calc_dst = WORK / 'physics_calculator_v2'
if calc_dst.exists():
    shutil.rmtree(calc_dst)
shutil.copytree(calc_src, calc_dst, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))

required_runtime = [WORK / 'physics_api_server.py', WORK / 'physics_pipeline.py', WORK / 'physics_engine_core.py', verified_file, calc_dst / 'calculator.py']
for path in required_runtime:
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f'Runtime file is missing or empty: {path}')
for path in [trace_adapter_dir, numeric_adapter_dir, chlt_adapter_dir]:
    for name in ['adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json']:
        if not (path / name).exists():
            raise RuntimeError(f'Adapter file missing: {path / name}')

print('WORK:', WORK)
print('API source:', server_file)
print('Pipeline source:', pipeline_file)
print('Core source:', core_file)
print('Calculator package:', calc_src)
print('Verified data:', verified_file)
print('Trace explainer adapter:', trace_adapter_dir)
print('Numeric parser adapter:', numeric_adapter_dir)
print('CHLT reasoner adapter:', chlt_adapter_dir)
print('\nOther verified files:', max(0, len(verified_hits) - 1))
print('Other trace adapter candidates:', max(0, len(trace_candidates) - 1))
print('Other numeric adapter candidates:', max(0, len(numeric_candidates) - 1))
print('Other CHLT adapter candidates:', max(0, len(chlt_candidates) - 1))


WORK: /kaggle/working/exact_physics_server
API source: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/physics_api_server.py
Pipeline source: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/physics_pipeline.py
Core source: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/physics_engine_core.py
Calculator package: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/physics_calculator_v2
Verified data: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/data/verified_golden_expanded.csv
Trace explainer adapter: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/qwen25_3b_trace_explainer_final_adapter/adapter
Numeric parser adapter: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/qwen25_3b_numeric_parser_final_adapter/adapter
CHLT reasoner adapter: /kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/qwen25_3b_chlt_reasoner_final_a

## 4. Optional Hugging Face Token

`Qwen/Qwen2.5-3B-Instruct` is public, but Kaggle may still need a Hugging Face token depending on account/cache settings. Add `HF_TOKEN` as a Kaggle Secret if model download fails.


## 5. Start vLLM Server

vLLM runs on `localhost:8001`. The LoRA adapter names used by the API are `trace_explainer_final`, `numeric_parser_final`, and `chlt_reasoner_final`.


In [4]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
VLLM_PORT = 8001
VLLM_LOG = Path('/kaggle/working/vllm_server.log')
if VLLM_LOG.exists():
    VLLM_LOG.unlink()

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', BASE_MODEL,
    '--host', '0.0.0.0',
    '--port', str(VLLM_PORT),
    '--enable-lora',
    '--max-lora-rank', '32',
    '--max-loras', '3',
    '--lora-modules',
    f'trace_explainer_final={trace_adapter_dir}',
    f'numeric_parser_final={numeric_adapter_dir}',
    f'chlt_reasoner_final={chlt_adapter_dir}',
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.85',
    '--dtype', 'half',
    '--trust-remote-code',
    '--enforce-eager',
]

print('Starting vLLM:')
print(' '.join(map(str, vllm_cmd)))
print('vLLM log:', VLLM_LOG)

vllm_log_fh = open(VLLM_LOG, 'a', encoding='utf-8')
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=vllm_log_fh,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)
print('vLLM PID:', vllm_proc.pid)


Starting vLLM:
/usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-3B-Instruct --host 0.0.0.0 --port 8001 --enable-lora --max-lora-rank 32 --max-loras 3 --lora-modules trace_explainer_final=/kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/qwen25_3b_trace_explainer_final_adapter/adapter numeric_parser_final=/kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/qwen25_3b_numeric_parser_final_adapter/adapter chlt_reasoner_final=/kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/qwen25_3b_chlt_reasoner_final_adapter/adapter --max-model-len 2048 --gpu-memory-utilization 0.85 --dtype half --trust-remote-code --enforce-eager
vLLM log: /kaggle/working/vllm_server.log
vLLM PID: 196


## 6. Wait For vLLM `/v1/models`

This cell waits until vLLM finishes loading. The first run can take several minutes because Kaggle may need to download and initialize the base model.


In [5]:
import urllib.request

def get_url(url, timeout=10):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode('utf-8', errors='replace')

def tail_file(path, n=80):
    path = Path(path)
    if not path.exists():
        return ''
    text = path.read_text(encoding='utf-8', errors='replace').splitlines()
    return "\n".join(text[-n:])

deadline = time.time() + 900
last_error = None
last_tail_print = 0
while time.time() < deadline:
    if vllm_proc.poll() is not None:
        print('vLLM process exited with code:', vllm_proc.returncode)
        print('Last vLLM log lines:')
        print(tail_file(VLLM_LOG, 120))
        raise RuntimeError('vLLM exited before /v1/models became ready')
    try:
        body = get_url(f'http://127.0.0.1:{VLLM_PORT}/v1/models', timeout=5)
        print(body[:2000])
        print('vLLM is ready')
        break
    except Exception as exc:
        last_error = exc
        now = time.time()
        if now - last_tail_print > 60:
            print('Still waiting for vLLM. Last error:', repr(last_error))
            print('Recent vLLM logs:')
            print(tail_file(VLLM_LOG, 50))
            last_tail_print = now
        time.sleep(5)
else:
    print('Last vLLM log lines:')
    print(tail_file(VLLM_LOG, 120))
    raise TimeoutError(f'vLLM did not become ready: {last_error}')


Still waiting for vLLM. Last error: URLError(ConnectionRefusedError(111, 'Connection refused'))
Recent vLLM logs:

Still waiting for vLLM. Last error: URLError(ConnectionRefusedError(111, 'Connection refused'))
Recent vLLM logs:
(APIServer pid=196) INFO 06-14 09:09:57 [api_utils.py:339] 
(APIServer pid=196) INFO 06-14 09:09:57 [api_utils.py:339]        █     █     █▄   ▄█
(APIServer pid=196) INFO 06-14 09:09:57 [api_utils.py:339]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.23.0
(APIServer pid=196) INFO 06-14 09:09:57 [api_utils.py:339]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-3B-Instruct
(APIServer pid=196) INFO 06-14 09:09:57 [api_utils.py:339]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=196) INFO 06-14 09:09:57 [api_utils.py:339] 
(APIServer pid=196) INFO 06-14 09:09:57 [api_utils.py:273] non-default args: {'lora_modules': [LoRAModulePath(name='trace_explainer_final', path='/kaggle/input/datasets/quanghuy225/full-api/physics_api_package/result/qwen25_3b_trace_explainer_final_adapter/

## 7. Warm Up LoRA Adapters

The first LoRA request on Kaggle T4 can be slow. This sends small warmup requests to all three adapters before the public API starts receiving traffic.


In [6]:
import urllib.request, json, time

def post_chat(model, content, timeout=180, max_tokens=48):
    req = urllib.request.Request(
        f'http://127.0.0.1:{VLLM_PORT}/v1/chat/completions',
        data=json.dumps({
            'model': model,
            'messages': [{'role': 'user', 'content': content}],
            'temperature': 0,
            'max_tokens': max_tokens,
        }).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read().decode('utf-8', errors='replace')

for model, prompt in [
    ('numeric_parser_final', 'Return parser JSON only: A 5 uF capacitor is connected to 20 V. Find charge.'),
    ('trace_explainer_final', 'Return JSON only with explanation for locked answer 100 uC from Q=C*U.'),
    ('chlt_reasoner_final', 'Return JSON only: In an ideal LC circuit with no resistance, is total energy conserved?'),
]:
    t0 = time.time()
    try:
        body = post_chat(model, prompt)
        print(model, 'warmup ok', round(time.time() - t0, 2), 's')
        print(body[:500])
    except Exception as exc:
        print(model, 'warmup failed:', repr(exc))
        print('vLLM may still work, but first API calls can timeout. Check /kaggle/working/vllm_server.log if this repeats.')


numeric_parser_final warmup ok 31.46 s
{"id":"chatcmpl-a4add408ce9247b3","object":"chat.completion","created":1781428288,"model":"numeric_parser_final","choices":[{"index":0,"message":{"role":"assistant","content":"{\"question_kind\":\"calculation\",\"topic\":\"capacitor\",\"given:[{'given_name':'U', 'given_unit':'V', 'given_value':20}, {'given_name':'C', 'given_unit':'uF', '","refusal":null,"annotations":null,"audio":null,"function_call":null,"tool_calls":[],"reasoning":null},"logprobs":null,"finish_reason":"length","stop_reason":nu
trace_explainer_final warmup ok 4.49 s
{"id":"chatcmpl-add5b4638173673b","object":"chat.completion","created":1781428320,"model":"trace_explainer_final","choices":[{"index":0,"message":{"role":"assistant","content":"Because the question asks for JSON only, I will not provide calculation steps, but rather the final locked evaluation: [{\"name\":\"given quantity\",\"role\":\"given_quantity\",\"raw\":\"100 uC\",\"value_si\":9.99","refusal":null,"annotations":

## 8. Start Physics Prediction API

The Flask API runs on `localhost:8000`. For grading speed, polish is disabled by default; numeric parser and CHLT reasoner remain enabled as fallbacks. You can set request `polish=true` for debugging.


In [18]:
API_PORT = 8000

def read_optional_secret(*names):
    for name in names:
        value = os.environ.get(name, '').strip()
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
            except Exception:
                value = ''
            if value:
                return str(value).strip()
    except Exception:
        pass
    return ''

# Optional: run the separate Type 1 notebook with Qwen2.5-Coder-7B first, then put its /predict URL here.
TYPE1_PREDICT_URL = 'https://ranging-thumb-encroach.ngrok-free.dev/predict'
TYPE1_LOCAL_MODEL = BASE_MODEL if not TYPE1_PREDICT_URL else 'Qwen/Qwen2.5-Coder-7B-Instruct'
print('Type1 7B proxy:', TYPE1_PREDICT_URL or 'not configured; using local 3B fallback for Type 1 smoke tests')

env = os.environ.copy()
env['VLLM_LOGGING_LEVEL'] = 'INFO'
env.update({
    'PHYSICS_API_HOST': '0.0.0.0',
    'PHYSICS_API_PORT': str(API_PORT),
    'PHYSICS_ENABLE_POLISH': 'false',
    'PHYSICS_POLISH_BASE_URL': f'http://127.0.0.1:{VLLM_PORT}/v1',
    'PHYSICS_POLISH_MODEL': 'trace_explainer_final',
    'PHYSICS_POLISH_TIMEOUT_SECONDS': '12',
    'PHYSICS_POLISH_MODE': 'conditional',
    'PHYSICS_POLISH_MAX_TOKENS': '120',
    'PHYSICS_POLISH_SEMANTIC': 'false',
    'PHYSICS_ENABLE_SEMANTIC_PARSER': 'true',
    'PHYSICS_SEMANTIC_MODEL': 'numeric_parser_final',
    'PHYSICS_SEMANTIC_MIN_CONFIDENCE': '0.50',
    'PHYSICS_SEMANTIC_REPAIR_THRESHOLD': '0.80',
    'PHYSICS_SEMANTIC_MAX_TOKENS': '512',
    'PHYSICS_SEMANTIC_TIMEOUT_SECONDS': '20',
    'PHYSICS_ENABLE_CHLT_REASONER': 'true',
    'PHYSICS_CHLT_MODEL': 'chlt_reasoner_final',
    'PHYSICS_CHLT_MAX_TOKENS': '256',
    'PHYSICS_CHLT_TIMEOUT_SECONDS': '20',
    'PHYSICS_ENABLE_TYPE1_LOGIC': 'true',
    'PHYSICS_TYPE1_MODEL': TYPE1_LOCAL_MODEL,
    'PHYSICS_TYPE1_BASE_URL': f'http://127.0.0.1:{VLLM_PORT}/v1',
    'PHYSICS_TYPE1_PROXY_URL': TYPE1_PREDICT_URL,
    'PHYSICS_TYPE1_MAX_TOKENS': '1024',
    'PHYSICS_TYPE1_TIMEOUT_SECONDS': '75',
    'PHYSICS_TYPE1_Z3_TIMEOUT_MS': '5000',
    'PHYSICS_VERIFIED_DATA': str(verified_file),
})

api_cmd = [sys.executable, str(WORK / 'physics_api_server.py')]
api_proc = subprocess.Popen(api_cmd, cwd=str(WORK), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('API PID:', api_proc.pid)

deadline = time.time() + 180
last_error = None
while time.time() < deadline:
    try:
        print(get_url(f'http://127.0.0.1:{API_PORT}/health', timeout=5)[:1500])
        print('API is ready')
        break
    except Exception as exc:
        last_error = exc
        time.sleep(3)
else:
    raise TimeoutError(f'API did not become ready: {last_error}')


Type1 7B proxy: https://ranging-thumb-encroach.ngrok-free.dev/predict
API PID: 556
{"chlt_max_tokens":256,"chlt_model":"chlt_reasoner_final","chlt_reasoner_enabled":true,"chlt_timeout_seconds":20.0,"default_polish_enabled":false,"default_polish_model":"trace_explainer_final","llm_fallback_enabled":false,"llm_load_4bit":true,"llm_local_files_only":true,"llm_model_error":null,"llm_model_name_or_path":"Qwen/Qwen2.5-Math-7B-Instruct","llm_model_ready":false,"ok":true,"pipeline_error":null,"pipeline_ready":true,"pipeline_version":"clean_blocks_with_verified_llm_json_guardrail","polish_base_url":"http://127.0.0.1:8001/v1","polish_max_tokens":120,"polish_mode":"conditional","polish_semantic_enabled":false,"polish_timeout_seconds":12.0,"prepared_data_path":"/kaggle/input/datasets/quanghuy225/full-api/physics_api_package/data/verified_golden_expanded.csv","project_root":"/kaggle/working","result_dir":"/kaggle/working/exact_physics_server","semantic_max_tokens":512,"semantic_min_confidence":0.5,

## 9. Test `/predict` Locally

This checks one Type 2 numeric query and one Type 1 logic query before opening the public tunnel.


In [19]:
payload = {
    'query_id': 'T2_KAGGLE_TEST_001',
    'type': 'type2',
    'query': 'An LC circuit consists of a 50 mH inductor and a 20 uF capacitor. If the maximum voltage across the capacitor is 12 V, what is the maximum current in the circuit?',
    'premises': [],
    'options': [],
    'debug': True,
    'polish': False,
}
req = urllib.request.Request(
    f'http://127.0.0.1:{API_PORT}/predict',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(req, timeout=60) as r:
    response = r.read().decode('utf-8')
print(response[:3000])
obj = json.loads(response)
assert isinstance(obj, list) and len(obj) == 1
assert obj[0]['query_id'] == payload['query_id']
assert obj[0]['answer']
assert obj[0]['explanation']
print('numeric method:', obj[0].get('debug', {}).get('method'))

type1_payload = {
    'query_id': 'TYPE1_LOGIC_LOCAL_TEST_001',
    'type': 'type1',
    'query': 'Does John receive academic distinction, according to the premises?',
    'premises': [
        'If a student completes all required courses, they are eligible for graduation.',
        'If a student is eligible for graduation and maintains a GPA above 3.5, they graduate with honors.',
        'If a student graduates with honors and completes a thesis, they receive academic distinction.',
        'John has completed all required courses.',
        'John maintains a GPA of 3.8.',
        'John has completed a thesis.',
    ],
    'options': [],
    'debug': True,
}
req = urllib.request.Request(
    f'http://127.0.0.1:{API_PORT}/predict',
    data=json.dumps(type1_payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(req, timeout=90) as r:
    type1_response = r.read().decode('utf-8')
print(type1_response[:2000])
type1_obj = json.loads(type1_response)
assert isinstance(type1_obj, list) and len(type1_obj) == 1
assert type1_obj[0]['query_id'] == type1_payload['query_id']
assert str(type1_obj[0]['answer']).strip() in {'Yes', 'No', 'Unknown'}
print('type1 method:', type1_obj[0].get('debug', {}).get('method'))


[{"answer":"0.24","debug":{"confidence":0.9,"elapsed_ms":106.31,"method":"formula_planner_lc_max_current","polish_error":null,"polish_status":"not_requested","prefix_pred":"DDT","semantic_model":null,"semantic_repair_from_confidence":null,"semantic_repair_from_method":null,"semantic_repair_status":null,"semantic_status":null,"topic_pred":"LC_oscillation"},"explanation":"The problem is treated as a ideal LC oscillation problem because it asks for current. The useful given data are 20.0 uF, 50.0 mH, 12.0 V. The required unit conversion is 20.0 uF -> 2e-05 F; 50.0 mH -> 0.05 H. The solver uses energy conservation: \u00bdCU\u2080\u00b2 = \u00bdLI\u2080\u00b2, so I\u2080 = U\u2080\u221a(C/L). Substitution and computation: Formula planner selected I\u2080 = U\u2080 \u00d7 \u221a(C/L). I\u2080=12\u00d7\u221a(2e-05/0.05)=0.24 A. Current is reported in amperes. Therefore, the final answer is 0.24 A.","premises_used":[],"query_id":"T2_KAGGLE_TEST_001","reasoning":{"steps":["Step 1: Identify the 

## 10. Expose Public Submission URL With ngrok

The portal requires the public `/predict` URL. Add `NGROK_AUTHTOKEN` as a Kaggle Secret before running this cell.


In [20]:
from pyngrok import ngrok

def read_secret_or_env(*names):
    for name in names:
        value = os.environ.get(name, '').strip()
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
            except Exception:
                value = ''
            if value:
                return str(value).strip()
    except Exception:
        pass
    return ''

NGROK_AUTHTOKEN = read_secret_or_env('NGROK_AUTHTOKEN', 'NGROK_TOKEN', 'GROK_AUTHTOKEN')
if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
else:
    print('No NGROK_AUTHTOKEN/NGROK_TOKEN found. Add it as a Kaggle Secret if ngrok requires authentication.')

ngrok.kill()
public_tunnel = ngrok.connect(addr=API_PORT, proto='http', bind_tls=True)
public_url = public_tunnel.public_url.rstrip('/')
prediction_url = public_url + '/predict'
vllm_models_url = public_url + '/v1/models'

print('Public API base:', public_url)
print('prediction_url=' + prediction_url)
print('vllm_models_url=' + vllm_models_url)

urls_txt = Path('/kaggle/working/urls.txt')
urls_txt.write_text(f'prediction_url={prediction_url}\nvllm_models_url={vllm_models_url}\n', encoding='utf-8')
print('Saved:', urls_txt)

Public API base: https://handwrite-matchless-enviable.ngrok-free.dev
prediction_url=https://handwrite-matchless-enviable.ngrok-free.dev/predict
vllm_models_url=https://handwrite-matchless-enviable.ngrok-free.dev/v1/models
Saved: /kaggle/working/urls.txt


## 11. Verify Public URLs

This cell checks the public health endpoint, public model visibility endpoint, and a public `/predict` call. Submit the printed `prediction_url` in the EXACT portal.


In [21]:
import urllib.error

def get_url_with_status(url, timeout=30):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status, r.read().decode('utf-8', errors='replace')
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode('utf-8', errors='replace')
    except Exception as e:
        return 'ERR', repr(e)

print('Local vLLM /v1/models:')
local_status, local_body = get_url_with_status(f'http://127.0.0.1:{VLLM_PORT}/v1/models', timeout=30)
print('status:', local_status)
print(local_body[:1500])

print('\nPublic /health:')
health_status, health_body = get_url_with_status(public_url + '/health', timeout=30)
print('status:', health_status)
print(health_body[:1500])

print('\nPublic /v1/models:')
models_status, models_body = get_url_with_status(vllm_models_url, timeout=30)
print('status:', models_status)
print(models_body[:1500])

public_payload = dict(payload)
public_payload['query_id'] = 'T2_PUBLIC_TUNNEL_TEST_001'
public_req = urllib.request.Request(
    prediction_url,
    data=json.dumps(public_payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
try:
    with urllib.request.urlopen(public_req, timeout=120) as r:
        public_response = r.read().decode('utf-8', errors='replace')
except urllib.error.HTTPError as e:
    public_response = e.read().decode('utf-8', errors='replace')
    print('\nPublic /predict HTTP error:', e.code)
    print(public_response[:3000])
    raise

print('\nPublic /predict:')
print(public_response[:3000])
public_obj = json.loads(public_response)
assert isinstance(public_obj, list) and len(public_obj) == 1
assert public_obj[0]['query_id'] == public_payload['query_id']
assert public_obj[0]['answer']
assert public_obj[0]['explanation']
print('\nPUBLIC API READY')


Local vLLM /v1/models:
status: ERR
URLError(ConnectionRefusedError(111, 'Connection refused'))

Public /health:
status: 200
{"chlt_max_tokens":256,"chlt_model":"chlt_reasoner_final","chlt_reasoner_enabled":true,"chlt_timeout_seconds":20.0,"default_polish_enabled":false,"default_polish_model":"trace_explainer_final","llm_fallback_enabled":false,"llm_load_4bit":true,"llm_local_files_only":true,"llm_model_error":null,"llm_model_name_or_path":"Qwen/Qwen2.5-Math-7B-Instruct","llm_model_ready":false,"ok":true,"pipeline_error":null,"pipeline_ready":true,"pipeline_version":"clean_blocks_with_verified_llm_json_guardrail","polish_base_url":"http://127.0.0.1:8001/v1","polish_max_tokens":120,"polish_mode":"conditional","polish_semantic_enabled":false,"polish_timeout_seconds":12.0,"prepared_data_path":"/kaggle/input/datasets/quanghuy225/full-api/physics_api_package/data/verified_golden_expanded.csv","project_root":"/kaggle/working","result_dir":"/kaggle/working/exact_physics_server","semantic_max_t

## 12. Keep Alive During Grading Slot

Run this cell during the registered grading slot so the notebook and tunnels stay alive. Stop it manually after grading is complete.


In [ ]:
print('Server is running. Keep this cell alive during your grading slot.')
try:
    while True:
        time.sleep(60)
        print(time.strftime('%H:%M:%S'), 'alive')
except KeyboardInterrupt:
    print('Stopped keep-alive loop.')

Server is running. Keep this cell alive during your grading slot.
09:27:34 alive
09:28:34 alive
09:29:34 alive
09:30:34 alive
09:31:34 alive
09:32:34 alive
09:33:34 alive
09:34:34 alive
09:35:34 alive
09:36:34 alive
09:37:34 alive
09:38:34 alive
09:39:34 alive
09:40:34 alive
09:41:34 alive
09:42:34 alive
09:43:34 alive
